## 1. zip() - pairing two lists

In [61]:
# zip() pairs items from several lists position by position
sensors = ["PD1" , "PD2","PD3"]      # list of sensor names (strings)
readings = [4.12 , 3.98 , 4.30]      # list of readings (floats), same order as sensors
# zip gives (name, value) pairs; the for loop unpacks each pair into two variables
for name , value in zip(sensors , readings):
    print(name, value)               # print sensor name and its reading

PD1 4.12
PD2 3.98
PD3 4.3


In [62]:
# zip() returns a lazy iterator, so wrap it in list()/dict() to see or keep the result
print(list(zip(sensors , readings)))   # list of tuples: [('PD1', 4.12), ...]
print(dict(zip(sensors , readings)))   # dict: first item of each pair = key, second = value

[('PD1', 4.12), ('PD2', 3.98), ('PD3', 4.3)]
{'PD1': 4.12, 'PD2': 3.98, 'PD3': 4.3}


In [63]:
# Lists of UNEQUAL length: 4 sensors but only 3 readings
sensors = ["PD1" , "PD2","PD3" , "PD4"]
readings = [4.12 , 3.98 , 4.30]

In [64]:
# zip() silently stops at the SHORTEST list, so PD4 is dropped with no error
# (be careful: missing data disappears quietly)
for name , value in zip(sensors , readings):
    print(name, value)

PD1 4.12
PD2 3.98
PD3 4.3


## 2. sorted(), max(), min() with key= and lambda

In [65]:
# A log is a list of tuples: (time, sensor, value). Tuples are fixed records
log = [("07:00:10", "PD2", 3.98), ("07:00:00", "PD1", 4.42), ("07:00:20", "PD3", 4.30)]

In [66]:
# sorted() returns a NEW sorted list; the original 'log' is not changed
print(sorted(log )) # by the first item (time) - tuples compare element by element
print(sorted(log , reverse=True)) # same, but descending (latest time first)
# key= takes a function that says WHAT to sort by; lambda is a tiny one-line function
# r is one tuple from the log, r[2] is its 3rd element (the reading value)
print(sorted(log , key=lambda r : r[2]))                 # ascending by reading
print(sorted(log , key=lambda r : r[2] , reverse=True))  # descending by reading
# max()/min() also accept key= and return the whole tuple with the biggest/smallest value
print("max :" , max(log , key=lambda r : r[2]))
print("min :" , min(log , key=lambda r : r[2]))

[('07:00:00', 'PD1', 4.42), ('07:00:10', 'PD2', 3.98), ('07:00:20', 'PD3', 4.3)]
[('07:00:20', 'PD3', 4.3), ('07:00:10', 'PD2', 3.98), ('07:00:00', 'PD1', 4.42)]
[('07:00:10', 'PD2', 3.98), ('07:00:20', 'PD3', 4.3), ('07:00:00', 'PD1', 4.42)]
[('07:00:00', 'PD1', 4.42), ('07:00:20', 'PD3', 4.3), ('07:00:10', 'PD2', 3.98)]
max : ('07:00:00', 'PD1', 4.42)
min : ('07:00:10', 'PD2', 3.98)


## 3. Dict & set comprehensions

In [67]:
# Dictionary of calibration offsets: sensor name -> offset to subtract from its reading
offsets = {"PD1": 0.12, "PD2": -0.05, "PD3": 0.03}

In [68]:
# Build a dict of raw readings (name -> value) from the two lists
raw = dict(zip(sensors, readings))

In [69]:
raw   # last line of a cell is displayed automatically

{'PD1': 4.12, 'PD2': 3.98, 'PD3': 4.3}

In [70]:
# DICT COMPREHENSION: {key: expression for ... in ...}
# .items() gives (key, value) pairs; we subtract each sensor's offset and round to 2 decimals
corrected = {name : round(value - offsets[name] , 2) for name , value in raw.items()}

In [71]:
corrected   # show corrected values

{'PD1': 4.0, 'PD2': 4.03, 'PD3': 4.27}

In [72]:
# Comprehension with a FILTER: only pairs where the condition (v > 4) is True are kept
above_4 = {name : v for name, v in corrected.items() if v > 4}

In [73]:
above_4   # PD1 (4.0) is dropped because 4.0 is not > 4

{'PD2': 4.03, 'PD3': 4.27}

In [74]:
# Channel numbers with duplicates
channels = [1,1,1,2,2]

In [75]:
# SET COMPREHENSION: {expression for ... in ...} - a set keeps only unique values
print({ch for ch in channels})            # {1, 2} - duplicates removed
print({ch for ch in channels if ch > 1})  # {2} - filter works here too

{1, 2}
{2}


## 4. Files and folders with pathlib

In [76]:
# pathlib.Path represents file/folder paths as objects (cleaner than plain strings)
from pathlib import Path

In [77]:
# Path to a folder named DATA (relative to the notebook's folder)
data_dir = Path("DATA")

In [78]:
# Create the folder; exist_ok=True means "no error if it already exists"
data_dir.mkdir(exist_ok=True)

In [79]:
# The / operator joins path parts: DATA + run1.txt -> DATA/run1.txt (works on any OS)
f = data_dir/"run1.txt"
f.write_text("hello world\n")   # creates/overwrites the file with this text in one call
print(f)

DATA\run1.txt


In [80]:
# .exists() returns True/False depending on whether the file is on disk
print("Exists:", f.exists())

Exists: True


In [81]:
# Path pieces: .name = full file name, .stem = name without extension, .suffix = extension
print("name:" , f.name, "| stem:", f.stem, "| suffix:", f.suffix)
print(f.read_text())   # read the whole file back as one string

name: run1.txt | stem: run1 | suffix: .txt
hello world



## 5. Reading a CSV with csv.DictReader

In [82]:
# csv module reads/writes comma-separated files
import csv

# newline="" is required with csv so line endings are handled correctly (no blank rows on Windows)
# 'with' closes the file automatically when the block ends
with open("./DATA/readings.csv",newline="") as f:
    reader = csv.DictReader(f) # first row becomes the keys; every later row becomes a dict
    for row in reader:         # one dict per data row
        print(row)             # all values are STRINGS (even numbers)
        print(row["channel"] , row["power_mw"])   # access a column by its header name

{'timestamp': '2026-10-27 07:00:00', 'instrument': 'PM100D', 'channel': '1', 'power_mw': '12.41', 'temp_c': '24.1'}
1 12.41
{'timestamp': '2026-10-27 07:00:05', 'instrument': 'PM100D', 'channel': '1', 'power_mw': '12.39', 'temp_c': '24.1'}
1 12.39
{'timestamp': '2026-10-27 07:00:10', 'instrument': 'PM100D', 'channel': '2', 'power_mw': '8.02', 'temp_c': '24.3'}
2 8.02
{'timestamp': '2026-10-27 07:00:15', 'instrument': 'PM100D', 'channel': '1', 'power_mw': 'ERR', 'temp_c': '24.2'}
1 ERR
{'timestamp': '2026-10-27 07:00:20', 'instrument': 'PM100D', 'channel': '2', 'power_mw': '8.05', 'temp_c': '24.4'}
2 8.05
{'timestamp': '2026-10-27 07:00:25', 'instrument': 'PM100D', 'channel': '1', 'power_mw': '12.47', 'temp_c': ''}
1 12.47
{'timestamp': '2026-10-27 07:00:30', 'instrument': 'PM100D', 'channel': '2', 'power_mw': '7.98', 'temp_c': '24.5'}
2 7.98


## 6. Cleaning data with try / except

In [83]:
# Two lists: rows that convert cleanly, and rows that fail
good , bad = [] , []

with open("./DATA/readings.csv", newline="") as f:
    # enumerate(..., start=2) numbers the rows; start=2 because line 1 is the header
    for line_no , row in enumerate(csv.DictReader(f) , start=2):
        try:
            # convert text to real numbers; float('ERR') or float('') raises ValueError
            row["power_mw"] = float(row['power_mw'])
            row["temp_c"] = float(row['temp_c'])
            row["channel"] = int(row['channel'])
            good.append(row)                       # reached only if ALL conversions worked
        except ValueError as e:
            bad.append((line_no , str(e)))         # remember which line failed and why

In [84]:
# Report: how many rows were fine, and why the others were skipped
print(len(good),"good rows")
for line_no , reason in bad:
    print(f"line {line_no} skipped : {reason}")   # f-string puts variables inside text

5 good rows
line 5 skipped : could not convert string to float: 'ERR'
line 7 skipped : could not convert string to float: ''


## 7. Writing a CSV with csv.DictWriter

In [85]:
# Write the cleaned rows to a new CSV ("w" = write mode, replaces the file if it exists)
with open("clean_readings.csv" , "w" , newline="") as f:
    # DictWriter needs the column order (fieldnames); this also lets us reorder columns
    writer = csv.DictWriter(f , fieldnames=["timestamp", "channel", "power_mw", "instrument" , "temp_c"])
    writer.writeheader()                  # write the header line first
    for row in good:
        # dict comprehension picks just the needed keys from each row, then writes one line
        writer.writerow({k : row[k] for k in ["timestamp", "channel", "power_mw" , "instrument" , "temp_c"]})

In [86]:
# Read the new file back to verify it (Path.read_text returns the whole file as a string)
print(Path("clean_readings.csv").read_text())

timestamp,channel,power_mw,instrument,temp_c
2026-10-27 07:00:00,1,12.41,PM100D,24.1
2026-10-27 07:00:05,1,12.39,PM100D,24.1
2026-10-27 07:00:10,2,8.02,PM100D,24.3
2026-10-27 07:00:20,2,8.05,PM100D,24.4
2026-10-27 07:00:30,2,7.98,PM100D,24.5



## 8. Grouping data and averaging per channel

In [87]:
# GROUPING: collect power values per channel
totals = {}
for row in good:
    # setdefault(key, []) returns the existing list for that channel, or creates an empty one first
    # then .append adds this row's power to that list
    totals.setdefault(row["channel"] , []).append(row["power_mw"])

In [88]:
totals   # {channel: [list of power values]}

{1: [12.41, 12.39], 2: [8.02, 8.05, 7.98]}

In [89]:
# sorted(totals.items()) orders by channel number; each item is (channel, list_of_values)
for ch,values in  sorted(totals.items()):
    # average = sum / count; :.3f formats to 3 decimal places
    print(f"Channel {ch} : avg {sum(values)/len(values):.3f} mW over {len(values)} readings")

Channel 1 : avg 12.400 mW over 2 readings
Channel 2 : avg 8.017 mW over 3 readings


## Real-life practice tasks

Use the `readings.csv` style data (timestamp, instrument, channel, power_mw, temp_c) and the concepts above.

1. **Sensor name lookup:** Given two lists `channels = [1, 2, 3]` and `labels = ["Pump", "Laser", "Fan"]`, use `zip` to print `Channel 1 -> Pump` style lines, then build a dict from them.
2. **Strongest readings:** Load the good rows from `readings.csv` and use `sorted` with a `lambda` key to print the top 3 rows by `power_mw`.
3. **Peak and dip:** Use `max` and `min` with a `key` to find the highest and lowest `temp_c` among the good rows and print their timestamps.
4. **Calibration:** Create an offsets dict per channel (e.g. `{1: 0.05, 2: -0.02}`) and use a dict comprehension to build a list/dict of corrected `power_mw` values.
5. **Alarm filter:** Use a comprehension with an `if` to keep only readings where `power_mw` is above 10 mW, and print how many there are.
6. **Unique instruments & channels:** Use a set comprehension to list every distinct channel found in `readings.csv`.
7. **Report folder:** Use `pathlib` to create a `REPORTS` folder, write a `summary.txt` inside it, then print its name, stem, suffix and whether it exists.
8. **Error report:** Extend the try/except loop so bad rows are written to a `rejected_readings.csv` file (line number, reason) using `csv.DictWriter`.
9. **Missing data:** Treat an empty `temp_c` as acceptable (use `None`) instead of rejecting the row, and report how many rows now pass.
10. **Per-channel statistics:** Using `setdefault`, group readings by channel and print the min, max and average power for each channel in a clean f-string table.


In [ ]:
# json module: save/load Python data as text (a standard format for config files)
import json

# A Python dict holding settings; nested dicts and lists are fine
config = {
    "station" : "Bench-2",                                                  # str
    "instrument" : {"model" : "PM100D" , "port" : "COM3", "timeout_s" : 2}, # nested dict
    "channels" : [1,2],                                                     # list
    "limits" : {"power_mw_min" : 8.0 , "power_mw_max" : 13.0},              # allowed power range
    "save_raw" : True                                                       # bool -> true in JSON
}

# "w" = write mode; 'with' closes the file automatically
with open("config.json", "w") as f:
    json.dump(config , f  , indent=2 ) # dump = Python object -> JSON file; indent makes it readable

# read the file back as plain text to see what was written
print(Path("config.json").read_text())

In [ ]:
# Read settings from the file instead of hard-coding them in the script
with open("config.json") as f:      # default mode is "r" (read)
    cfg = json.load(f)              # load = JSON file -> Python dict

# Index into nested dicts with chained keys
lo = cfg["limits"]["power_mw_min"]  # lower limit (8.0)
hi = cfg["limits"]["power_mw_max"]  # upper limit (13.0)

for row in good:                    # 'good' = cleaned rows from section 6
    # chained comparison: True only when lo <= power <= hi; ternary picks the label
    status = "Pass" if lo <= row["power_mw"] <=  hi else "Fail"
    # [-8:] slices the last 8 chars of the timestamp -> just the time HH:MM:SS
    print(row["timestamp"][-8 : ] , row["channel"] , row["power_mw"] , status)

In [ ]:
# logging: timestamped, levelled messages instead of scattered print() calls
# levels (low -> high): DEBUG < INFO < WARNING < ERROR < CRITICAL
import logging

In [ ]:
# basicConfig sets up logging ONCE (later calls are ignored unless the handlers are removed)
logging.basicConfig(
    level = logging.INFO,                                  # show INFO and above; hide DEBUG
    format= "%(asctime)s %(levelname)-8s %(message)s",     # time, level padded to 8 chars, text
    datefmt="%H:%M:%S",                                    # time style: hours:minutes:seconds
    handlers = [                                           # handlers = where the messages go
        logging.FileHandler("run.log"), # save to a file
        logging.StreamHandler(), #  and show on screen
    ]
)

logging.debug("raw bytes from meter : b'12.41\\r\\n'") # hidden (below INFO)
logging.info("Test started on Bench-2")                # normal progress message
logging.warning("Temperature reading missing on line 7") # something odd, but the run continues
logging.error("Could not convert 'ERR' to float on line 5") # an operation failed
logging.info("Test Finished")

In [ ]:
# NOTE: this basicConfig is ignored in the same session (logging was already configured above)
logging.basicConfig(level=logging.INFO , format= "%(levelname)s %(message)s")

try:
    value = float("ERR")            # raises ValueError: 'ERR' is not a number
except ValueError:
    # logging.exception = ERROR message + the full traceback (use only inside an except block)
    logging.exception("Bad reading")
# the error was handled, so the script carries on
logging.info("Script keeps running")